# Описание проекта

**Описание проблемы:**

Каждый банк стремится удержать своих клиентов для поддержания своего бизнеса. Хорошо известно, что привлечение нового клиента обходится дороже, чем удержание существующего.

Для банков важно понимать причины, которые приводят клиента к решению покинуть компанию.

Предотвращение оттока позволяет банкам разрабатывать программы лояльности и кампании по удержанию, направленные на сохранение максимального числа клиентов.

Ниже приведены данные о клиентах-владельцах счетов в анонимном международном банке, цель анализа которых заключается в предсказании оттока клиентов.

**О наборе данных** 

Нам предоставлен датасет в виде файла _Customer-Churn.csv_, со следующим набором признаков:

`RowNumber` — соответствует номеру записи (строки).

`CustomerId` — содержит уникальные случайные значения, id клиента.

`Surname` — фамилия клиента.

`CreditScore` — кредитный рейтинг клиента.

`Geography` — местоположение клиента.

`Gender` — пол клиента.

`Age` — возраст.

`Tenure` — количество лет, в течение которых клиент является клиентом банка.

`Balance` — баланс на счетах клиента.

`NumOfProducts` — количество используемых клиентом продуктов банка.

`HasCrCard` — наличие у клиента кредитной карты.

`IsActiveMember` — признак активного клиента.

`EstimatedSalary` — прогнозируемый доход.

`Exited` — покинул ли клиент банк.

`Complain` — наличие у клиента жалобы.

`Satisfaction Score` — оценка, предоставляемая клиентом за разрешение его жалобы.

`Card Type` — тип карты, которой владеет клиент.

`Point Earned` — баллы, начисленные клиенту за использование кредитной карты.

**Задачи проекта:**

1. Выполнить анализ данных оттока клиентов. Составить портрет наиболее вероятного клиента склонного к уходу из банка. На основе анализа подготовить рекомендации для отдела маркетинга.
1. Используя методы машинного обучения, разработать модель для прогнозирования вероятности уйдет ли клиент из банка в ближайшее время.
1. Оценить качество полученной модели.

**Порядок выполнения работы:**

1. Загрузим данные и импортируем необходимые библиотеки.
1. Проведем предобработку данных, проверим корректность типов, наименование признаков, сбалансированность классов целевого признака, количество пропусков и дубликатов, оценим типичность и не типичность количественных значений (выбросы). Устраним выявленные проблемы.
1. Выполним исследовательский анализ, оценим статистику влияния признаков на отток, построим графики для визуализации. Сформулируем гипотезы и проверим их.
1. На основе результатов анализа составим наиболее вероятные портреты лояльного клиента и клиента склонного к оттоку. Подготовим рекомендации по удержанию оттока.
1. Подготовим данные для моделирования, определим категориальные и числовые признаки, выберем значимые признаки, выделим целевой признак, создадим обучающую и тестовую выборки, выполним кодирование.
1. Создадим две разные модели для классификации клиентов - уйдет/не уйдет. Оценим качество моделей и выберем лучшую.
1. Подведем общие итоги проекта и сделаем выводы.

**Требования к выполнению проекта:**

- Технологический стек: `python`, `numpy`, `pandas`, `matplotlib`, `seaborn`, `sklearn`.
- Имена переменным и признакам присваиваем в стиле _snake_case_.
- При написании кода придерживаемся правил _PEP8_ (https://pythonworld.ru/osnovy/pep-8-rukovodstvo-po-napisaniyu-koda-na-python.html), код комментируем. При оформлении текста используем разметку _markdown_ (https://gist.github.com/Jekins/2bf2d0638163f1294637).
- Датасет делим на обучающую и тестовую выборки в соотношении 75:25 с учетом стратификации целевого признака `Exited`.
- Для моделирования используем классические ML алгоритмы: случайный лес и логистическая регрессия.
- Метрики для оценки качества моделей: ROC AUC и матрица ошибок.
- Обучение моделей выполняем с подбором гиперпараметров, а для усреднения результатов используем кроссвалидацию.

# Загрузка датасета и первичный осмотр данных

In [ ]:
import pandas as pd
import numpy as np
import phik
import joblib

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, precision_recall_curve, brier_score_loss
from sklearn.calibration import CalibratedClassifierCV, calibration_curve

from sklearn.preprocessing import LabelEncoder

from datetime import datetime

from optbinning import OptimalBinning, scorecard
import pandas as pd

#подгпрузка данных
df = pd.read_csv('Customer-Churn.csv')

#просмотр первых пяти строк, чтобы убедиться, что мы все правильно подгрузили 
df.head()

In [ ]:
#рассмотрим общую информацию о типах данных и наличии пропусков 
df.info()

# Чистка и предобработка

Мы имеем CreditScore (3126 пропусков), IsActiveMember (770 пропусков), EstimatedSalary (2459 пропусков), Complain (770 пропусков), Satisfaction Score (770 пропусков), также имеются неинформативные столбцы (RowNumber, CustomerId, Surname), их удалим

In [ ]:
#удаление неинформативных признаков:
df.drop(columns=['RowNumber', 'CustomerId', 'Surname'], inplace=True, errors='ignore')

## CreditScore

Заменим пропуски медианой

In [ ]:
credit_median = df['CreditScore'].median()
df['CreditScore'] = df['CreditScore'].fillna(credit_median)

In [ ]:
#проверим отсутствие пропусков
df.CreditScore.isnull().sum()

## EstimatedSalary

In [ ]:
df.EstimatedSalary.describe()

In [ ]:
df = df[df['EstimatedSalary'] > 0]

Медиана у данного признака 101371, заполним ею пропуски

In [ ]:
#находим медиану
median_salary = df.EstimatedSalary.median()
print(f'Mедиана EstimatedSalary: {median_salary:,.0f}')

#заполним пропуски медианой
df.EstimatedSalary = df.EstimatedSalary.fillna(median_salary)
#проверим результат
print('Пропусков в EstimatedSalary после обработки:', df.EstimatedSalary.isnull().sum())

In [ ]:
#Проверим выбросы в EstimatedSalary
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize = (6,8))
sns.boxplot(y = df['EstimatedSalary'])
plt.title('Выбросы в EstimatedSalary')
plt.show()

In [ ]:
#удaлим выбросы 

Q1 = df['EstimatedSalary'].quantile(0.25)
Q3 = df['EstimatedSalary'].quantile(0.75)
IQR = Q3 - Q1

#определяем ограницы
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

#фильтруем датасет
df = df[(df['EstimatedSalary'] >= lower_bound) & (df['EstimatedSalary'] <= upper_bound)]

## IsActiveMember

In [ ]:
print('Тип данных: ', df.IsActiveMember.dtype)
print('Колиество пропусков: ', df.IsActiveMember.isnull().sum())
print('Уникальные значения: ', df.IsActiveMember.unique())

Тип данных float - не подходящий для бинарной переменной, мы сможем изменить его на int() после обработки пропусков.

In [ ]:
mode_val = df['IsActiveMember'].mode()[0]
df['IsActiveMember'] = df['IsActiveMember'].fillna(mode_val)
df['IsActiveMember'] = df['IsActiveMember'].astype(int)

In [ ]:
#проверим наличие пропусков и тип IsActiveMember
print('Тип данных признака IsActiveMember:', df.IsActiveMember.dtypes)
print('Количество пропусков в признаке IsActiveMember:', df.IsActiveMember.isna().sum())

## Age

In [ ]:
#Проверим данные
print('Тип данных: ', df.Age.dtype)
print('Колиество пропусков: ', df.Age.isnull().sum())
print('Описание:')
df.Age.describe()

По описательной статистике явно видно, что в признаке Age есть аномалии, проверим сколько таких значений

In [ ]:
#Сколько клиентов старше 100 лет?
anomalous_age = df[df['Age'] > 100]
print(f'количество клиентов старше 100 лет: {len(anomalous_age)}')

Удалим аномалии: клиенты старше 100 лет

In [ ]:
df = df[df['Age'] <= 100]

In [ ]:
plt.figure(figsize=(10,6))
sns.histplot(df['Age'], bins = 30,kde = True, color = 'purple')

plt.show()

## Complain

In [ ]:
print('всего пропусков',df['Complain'].isnull().sum())
print('процент пропусков от всех данных:',df['Complain'].isnull().mean()*100)

In [ ]:
df = df.dropna(subset = ['Complain'])

In [ ]:
print('всего пропусков',df['Complain'].isnull().sum())

## SatisfactionScore

In [ ]:
df['Satisfaction Score'].isna().sum()

Пропусков нет после удаления пропусков признака Complain

# Разведовочный анализ данных (EDA)

Посмотрим общий отток по базе

In [ ]:
total_churn = df['Exited'].mean()
print(f'Доля ушедших клиентов от всего числа: {total_churn:1%}')

Получается, 20% клиентов из датасета ушли

In [ ]:
#Распределение целевой переменной Exited:
plt.figure(figsize = (6,4))
df['Exited'].value_counts().plot(kind = 'bar', color = ['purple', 'salmon'], edgecolor = 'black')
plt.title('Распределение целевой переменной')
plt.xticks(ticks = [0,1], labels = ['остался (0)', 'ушел (1)'], rotation = 0)
plt.ylabel('количество клиентов')
for i, v in enumerate(df['Exited'].value_counts()):
    plt.text(i, v + 50, str(v), ha = 'center', va = 'bottom')
plt.show()

In [ ]:
#Построитм тепловую карту на числовых признаках
numeric_cols = ['CreditScore', 'Age', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Complain', 'Satisfaction Score', 'Point Earned', 'Tenure', 'Exited']

In [ ]:
# вычисляем матрицу phi_k корреляций
phik_matrix = df[numeric_cols].phik_matrix()

#визуализируем
plt.figure(figsize = (10, 6))
sns.heatmap(phik_matrix, annot = True, cmap = 'coolwarm', center = 0, square = True, linewidths = .5, fmt = '.2f')
plt.title('матрица корреляций phi_k');

На основании тепловой карты мы видим, что явная зависимость наблюдается между Complain и Exited, получается, жалоба -> почти гарантированный отток.
Также видно, что есть зависимость возраста, количества продуктов и оттока.

Теперь рассмотрим ключевые признаки

## EDA IsActiveMember

In [ ]:
#сравним уровень оттока по группам
#группируем по флагу пропуска:
churn_by_missing = df.groupby('IsActiveMember')['Exited'].agg(
    count = 'size',
    churn_rate = 'mean' #доля ушедших клиентов
).round(3)

print(churn_by_missing)

Итог проверки:

*неактивные клиенты уходят с вероятностью 24.5%

*активные клиенты уходят с вероятностью 15.9%
Неактивные клиенты уходят на 8.6% чаще,чем активные

Это значимая разница - значит, признак IsActiveMember важен для модели

## EDA Satisfaction Score и Complain

In [ ]:
#связь с оттоком и Complain
churn_by_Complain = df.groupby('Complain')['Exited'].agg(
    count = 'size',
    churn_rate = 'mean' #доля клиентов, которые ушли
).round(3)
print(churn_by_Complain)

Из тех, кто пожаловался 77% в дальнейшем перестал быть клиентом банка. Из тех, кто не жаловался, ушло всего 5%.

In [ ]:
#связь с оттоком и Satisfaction Score
churn_by_Satisfaction = df.groupby('Satisfaction Score')['Exited'].agg(
    count = 'size',
    churn_rate = 'mean' #доля клиентов, которые ушли
).round(3)
print(churn_by_Satisfaction)

Значимых отличий нет, уровень оттока почти не зависит от оценки

In [ ]:
#отфильтруем клиентов, у которых Complain = 1
complain_mask = df['Complain'] == 1
df_complain = df[complain_mask].copy()

#посчитаем среднюю оценку среди тех, кто жаловался и оценил
mean_satisfaction_complain = df_complain['Satisfaction Score'].mean()
print(f'средняя оценка решения жалобы(среди жаловавшихся):{mean_satisfaction_complain:2f}')

#разделим жаловавшихся на группы по оценке

#группа А: жаловались и оценили
df_rated = df_complain[df_complain['Satisfaction Score'].notna()].copy()

#добавим признак: ниже средней оценки?
df_rated['Below_Average_Score'] = df_rated['Satisfaction Score'] < mean_satisfaction_complain

#группа B: жаловались, но не оценили
df_not_rated = df_complain[df_complain['Satisfaction Score'].isna()]

#посчитаем долю ушедших 

# 1. жаловались и оценили ниже среднего
churn_rated_low = df_rated[df_rated['Below_Average_Score']]['Exited'].mean()
count_rated_low = len(df_rated[df_rated['Below_Average_Score']])

# 2. жаловались и оценили выше или равно среднему
churn_rated_high = df_rated[~df_rated['Below_Average_Score']]['Exited'].mean()
count_rated_high = len(df_rated[~df_rated['Below_Average_Score']])

# 3.жаловались, но не оценили
churn_not_rated = df_not_rated['Exited'].mean()
count_not_rated = len(df_not_rated)

#вывод результатов
print('анализ оттока среди клиентов, подавших жалобу (Complain = 1):')
print('-' * 60)
print(f'оценили и ниже среднего ({count_rated_low} чел.): {churn_rated_low:.2%} ушли')
print(f'оценили и выше\равно среднего ({count_rated_high} чел.): {churn_rated_high:.2%} ушли')
print(f'не оценили ({count_not_rated} чел.): {churn_not_rated:.2%} ушли')

In [ ]:
#визуализируем полученные результаты
summary = pd.DataFrame({
    'Group': [
        'оценили и ниже среднего',
        'оценили и выше или равно среднего',
        'не оценили'
    ],
    'Churn Rate': [
        churn_rated_low,
        churn_rated_high,
        churn_not_rated
    ],
    'Count':[
        count_rated_low,
        count_rated_high,
        count_not_rated
    ]
})
plt.figure(figsize = (10,6))
sns.barplot(data=summary, x = 'Group', y = 'Churn Rate', palette = 'Set3', hue='Group')
plt.title('Доля ушедших  среди клиентов, подавших жалобу')
plt.ylabel('Доля ушедших')
plt.xlabel('Группа')
plt.xticks(rotation = 15)
for i, row in summary.iterrows():
    plt.text(i,row['Churn Rate'] + 0.01, f"{row['Churn Rate']:.1%}", ha = 'center')
plt.show()

Вывод: почти одинаковый отток в обеих группах среди жаловавшихся

оценили ниже среднего - 78.3% ушли 

оценили выше\равно среднего - 76.6% ушли
Факт жалобы сам по себе - главный фактор оттока, а не оценка ее решения

Разница всего в 2% - очень маленькая, а значит:

Даже если банк решит проблему и клиент оценил решение высоко, он почти так же часто уходит, как и тот, кто оценил плохо (то есть жалоба = точка невозврата для большинства)

Влияние на модель:

Satisfaction Score слабо влияет на отток среди жаловавшихся

Самый сильный признак - наличие жалобы

## EDA Geography

In [ ]:
print('распределение клиентов по месту проживания')
df['Geography'].value_counts()

Основная часть клиентов из Франции, другая часть поделилась пополам на Германию и Испанию

In [ ]:
#связь с оттоком
churn_by_card = df.groupby('Geography')['Exited'].agg(
    count = 'size',
    churn_rate = 'mean' #доля клиентов, которые ушли
).round(3)
print(churn_by_card)

Клиенты из Германии уходят значительно чаще, чем из Франции или Испании

связь с другими признаками:

In [ ]:
#Создадим временные группы
bins = [-1, 0, 50000, 100000, 150000, 200000, 1000000]
labels = ['Zero', 'Low', 'Medium', 'High', 'Very_High', 'Extreme']

df['Balance_group'] = pd.cut(df['Balance'], bins = bins, labels = labels)

#отток по странам и балансу:
df.pivot_table(
    index = 'Geography',
    columns = 'Balance_group',
    values = 'Exited',
    aggfunc = 'mean',
    observed=False
).round(3)

В Германии:

высокий отток в группе High (100-150k) 36%

Даже в zero - 18.8%, выше, чем в других странах

В Extreme - 26% (ниже, чем во Франции)

Во Франции:

Максимальный отток у Extreme (44.9%) (возможно уходят в приватные банки)

В Испании:

Высокий отток в Very_High и  в Extreme

In [ ]:
#страна и количество продуктов:
pd.crosstab(df['Geography'], df['NumOfProducts'], normalize = 'index').round(3)

В Германии:

больше всего клиентов с 1 продуктом (52.3%)

доля с 3 продуктами - 3.6%, что выше, чем в других странах

In [ ]:
#страна и жалобы:
df.groupby('Geography')['Complain'].mean().round(3)

В Германии в 1.8 раза больше жалоб, чем во Франции. Это является главным сигналом, проблема оттока в Германии не в продуктах или балансе, а в качестве сервиса.

In [ ]:
#страна и удовлетворенность после жалобы
df[df['Complain'] == 1].groupby('Geography')['Satisfaction Score'].mean()

Различие незначительное

Итоги анализа Geography:

❗❗Германия - ключевой очаг проблемы оттока.

Причины:

1. Высокая доля жалоб(30% клиентов)

2. Низкая удовлетворенность после жалоб (2.99)

3. Рост оттока у клиентов с 3 продуктами и высоким балансом

4. Высокий отток даже среди тех, кто с 1 продуктом

## EDA NumOfProducts

In [ ]:
print('уникальные значения:', df['NumOfProducts'].unique())
print('\nРаспределение значений:')
print(df['NumOfProducts'].value_counts().sort_index())

Половина клиентов с 1 продуктом, с 2 клиентами почти вторая половина и с 3-4 продуктами - единицы.

In [ ]:
#визуализируем
df['NumOfProducts'].value_counts().sort_index().plot(
    kind = 'bar', color = 'purple', edgecolor = 'black'
)
plt.title('распределение количества продуктов')
plt.xlabel('количество продуктов')
plt.ylabel('число клиентов')
plt.xticks(rotation = 0)
plt.show()

Выставим гипотезу: чем больше продуктов, тем выше лояльность Проверим связь с оттоком

In [ ]:
churn_by_products = df.groupby('NumOfProducts')['Exited'].agg(
    count = 'size',
    churn_rate = 'mean'
).round(3)
print(churn_by_products)

In [ ]:
#визуализация
plt.figure(figsize = (8,5))
sns.barplot(data = df, x = 'NumOfProducts', y = 'Exited', errorbar = None)
plt.title('доля оттока по количеству продуктов')
plt.ylabel('доля ушедших')
plt.xlabel('количество продуктов')
plt.show()

Построим тепловую карту той же зависимости, что и выше

In [ ]:
pivot = df.pivot_table(index = 'NumOfProducts', columns = 'IsActiveMember', values = 'Exited', aggfunc = 'mean')
sns.heatmap(pivot, annot = True, cmap = 'Reds', center = 0.2)
plt.title('Отток по количеству продуктов и активности')
plt.show()

Самый большой отток у неактивных клиентов с 3-4 продуктами, что еще может быть логично, набрали продуктов и перестали пользоваться --> ушли

А вот высокий отток среди активных клиентов с 4 продуктами уже странно

Очень важный признак для модели

In [ ]:
pd.crosstab(df['NumOfProducts'], df['IsActiveMember'], normalize = 'index').round(3)

Значимой разницы нет

Рассмотрим связь с жалобами и оттоком

In [ ]:
churn_by_complain = df.groupby(['NumOfProducts','Complain'])['Exited'].agg(
    count = 'size',
    churn_rate = 'mean'
).round(3)
print(churn_by_complain)

Клиенты с 2-мя продуктами самые лояльные: при наличии жалобы уходят в 60% случаев, при остальных количествах продуктов клиент при наличии жалобы уйдет при вероятности более 80%

Распределение клиентов по странам и количеству продуктов:

In [ ]:
crosstab_geo = pd.crosstab(df['NumOfProducts'], df['Geography'], normalize = 'index')*100
crosstab_geo.round(1)

✔Франция - лидер по охвату

41-50%% всех клиентов с 1-4 продуктами

✔Германия - растет с количеством продуктов

у клиентов с 1  продуктом 27%, с 3 продуктами 33.8%

✔Испания - не лидер

всегда примерно 24% клиентов, неза/висимо от количества продуктов
Ранее мы заметили что в Германии самый высокий отток: 27,5%, во Франции и в Испании 17,3% и 18,7% соотвенственно.

Высокий отток в Германии - не потому, что у них больше продуктов, возможно в Германии высокий отток из-за качества сервиса или проблем удержания после продаж



Ключевые выводы по NumOfProducts:

❗Клиенты с 3-4 продуктами уходят в 60% случаев

❗Больше всего продуктами (1-4) интересуются во Франции(50% при любом количестве продуктов)|

## EDA Balance

In [ ]:
#описательная статистика:
df['Balance'].describe()

In [ ]:
#распределение: гистограмма

plt.figure(figsize = (12, 6))
sns.histplot(df['Balance'], bins = 50, color = 'purple', edgecolor = 'black')
plt.title('Распределение баланса на счете')
plt.xlabel('Баланс')
plt.ylabel('Число клиентов')
plt.axvline(x = 0, color = 'red', linestyle = '--', label = 'Нулевой баланс')
plt.legend()
plt.show()

Гистограмма Balance имеет резкий пик на нуле(~4900 клиентов), что соответствует неактивным клиентам. Далее, от 0 до 120 000 баланса, количество клиентов плавно растет, достигая максимума(1000), после чего плавно стремится к нулю при балансе ~200000.

Пик на 120 000 - оптимальный уровень сбережений

Падение после - мало богатых клиентов.
Распределение не нормальное, но отражает реальную структуру клиентов.

In [ ]:
#Создадим временные группы
bins = [-1, 0, 50000, 100000, 150000, 200000, 1000000]
labels = ['Zero', 'Low', 'Medium', 'High', 'Very_High', 'Extreme']

df['Balance_group'] = pd.cut(df['Balance'], bins = bins, labels = labels)

#теперь посмотрим кто чаще уходит
churn_by_balance = df.groupby('Balance_group', observed=False)['Exited'].agg(
    count = 'size',
    churn_rate = 'mean'
).round(3)

churn_by_balance

Наблюдается интересная зависимость: чем выше баланс, тем больше отток. Построим визуализацию

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize = (10,6))
sns.barplot(data = df, x = 'Balance_group', y = 'Exited')
plt.title('Доля оттока по уровням баланса')
plt.ylabel('Доля ушедших')
plt.xticks(rotation = 0)
plt.show()

Действительно, форма зависимости - не U-образная, а возрастающая в конце, пик на Extreme.

Клиенты с самым высоким балансом уходят больше чем в 2 раза чаще чем те, у кого нулевой баланс.

Это может указывать на проблему удержания богатых клиентов


Посмотрим отток по балансу и активности

In [ ]:
pivot = df.pivot_table(
    index = 'Balance_group',
    columns = 'IsActiveMember',
    values = 'Exited',
    aggfunc = 'mean', 
    observed=False
)
sns.heatmap(pivot, annot = True, cmap = 'Reds', center = 0.2)
plt.title('Отток по балансу и активности')
plt.show()

Интересный момент, который может подтверждать проблему, поднятую выше: основная часть из ушедших это клиенты с большим балансом и положительной активностью

In [ ]:
pd.crosstab(df['Balance_group'], df['Geography'], normalize = 'index').round(3)

Клиенты с самым высоким балансом из Германии(51.1%), при этом, в другим группах Германия 2%-36%. В Германии не просто высокий отток, а высокий отток у самых богатых и активных клиентов

Итог по Balance:

Распределение необычное

резкий пик на нуле, затем плавный рост до 120 000 и плавное падение

❗2) Самый высокий отток - у клиентов с экстремально высоким балансом (32% - каждый третий уходит)

Особенно среди активных клиентов из Германии

## EDA Age

In [ ]:
df['Age'].describe()

Средний возраст - 38 лет, разброс умеренный.

In [ ]:
plt.figure(figsize = (10,6))
sns.histplot(df['Age'], bins = 30, kde = True, color = 'pink', edgecolor = 'purple')
plt.title('Распределение возраста клинтов')
plt.xlabel('Возраст')
plt.ylabel('Число клиентов')
plt.show()

Пик в районе 30-40 - типично для массового банка. Хвост вправо - есть возрастные клиенты.

In [ ]:
#группировка по возрасту
bins = [18, 30, 40 ,50, 60, 100]
labels = ['18-30', '31-40', '41-50', '51-60', '60+']
df['Age_group'] = pd.cut(df['Age'], bins = bins, labels = labels)

#отток по возрастным группам
churn_by_age = df.groupby('Age_group', observed=False)['Exited'].agg(
    count = 'size',
    churn_rate = 'mean'
).round(3)

churn_by_age


U-образной зависимости нет.

У молодых клиентов отток минимальный, а далее с ростом возраста растет и отток. У клиентов возрастной группы 51-60 отток 46%(почти половина уходит) - самая рискованная группа.

Посмотрим связь с другими признаками

In [ ]:
df.pivot_table(index = 'Age_group', columns = 'IsActiveMember', values = 'Exited', aggfunc = 'mean', observed=False).round(3)

❗❗У неактивных (0) отток катастрофически высок:

В 51-60: 68.9%

В 60+: 67.2%

У активных (1) - отток в 2-3 раза ниже, но все равно растет с возрастом

У клиентов 50+ баланс чуть выше (на 5-10 тыс)

In [ ]:
corr = df['Age'].corr(df['Tenure'])
corr

связи нет

In [ ]:
#связь с количествоим продуктов
df.groupby('Age_group', observed=False)['NumOfProducts'].mean().round(0)

18-50 лет: в среднем 2 продукта -> вовлеченность.

51-60+ лет: в реднем 1 продукт -> низкая вовлеченность.

Возрастные клиенты:

1. используют только один продукт

2. часто неактивны

3. уходят в 60-64% случаев, если неактивны.

Ключевой фактор риска - не сам возраст, а сочетание:

✔возраст 50+

✔один продукт

✔низкая активность
Клиенты этой группы мало вовлечены, не используют преимущества банка, и массово уходят.

## EDA CreditScore

In [ ]:
df['CreditScore'].describe()

In [ ]:
plt.figure(figsize = (10,6))
sns.histplot(df['CreditScore'], bins = 50, color = 'purple', edgecolor = 'black')
plt.title('Распределение CreditScore')
plt.xlabel('CreditScore')
plt.ylabel('Число клиентов')
plt.axvline(x = 651, color = 'yellow', linestyle = '--', label = 'Медиана')
plt.legend()
plt.show()

Можно заметить пик на медиане(651) - это искуственный пик из-за заполнения пропусков

Форма в целом нормальная, но с плечом слева

In [ ]:
#группировка по уровням рейтинга

#временные группы
bins = [300, 580, 670, 740, 850]
labels = ['Low (<580)', 'Fair (580 -669)', 'Good(670-739)', 'Excellent (740+)']
df['CreditScore_group'] = pd.cut(df['CreditScore'], bins = bins, labels = labels)

#отток по группам
churn_by_credit = df.groupby('CreditScore_group', observed=False)['Exited'].agg(
    count = 'size',
    churn_rate = 'mean'
).round(3)

print('отток по уровню кредитного рейтинга: ')
print(churn_by_credit)

Наибольший отток у клиентов с низким рейтингом (21%). Наименьший - у тех, у кого хороший рейтинг(670 - 739: 18.1%)

In [ ]:
#Связь с Balance
df.groupby('CreditScore_group', observed=False)['Balance'].mean().round(2)

Самый высокий баланс у группы Fair, у остальных ~75-81к. Клиенты со средним рейтингом имеют наибольший баланс, возможно, это основной сегмент банка.

In [ ]:
#Связь с NumOfProducts
df.groupby('CreditScore_group', observed=False)['NumOfProducts'].mean().round(2)

Количество продуктов одинаковое, нет связи рейтинга с вовлеченностью

In [ ]:
#Связь с Complain
df.groupby('CreditScore_group', observed=False)['Complain'].mean().round(2)

Жалобы распределены равномерно - не зависят от уровня кредитного рейтинга

## EDA EstimatedSalary

In [ ]:
df['EstimatedSalary'].describe()

In [ ]:
plt.figure(figsize = (10,6))
sns.histplot(df['EstimatedSalary'], bins = 50, color = 'purple', edgecolor = 'black')
plt.title('Распределение EstimatedSalary')
plt.xlabel('Оценка дохода')
plt.ylabel('Число клиентов')
plt.axvline(x = 0, color = 'yellow', linestyle = '--', label = 'нулевой доход')
plt.legend()
plt.show()

In [ ]:
#отток по группам дохода:
bins = [-40000, 0, 50000, 100000, 150000, 200000]
labels = ['Negative', 'Zero', 'Low', 'Medium', 'High']
df['Salary_group'] = pd.cut(df['EstimatedSalary'], bins = bins, labels = labels)

#посмотрим на отток:
#отток по группам
churn_by_salary = df.groupby('Salary_group', observed=False)['Exited'].agg(
    count = 'size',
    churn_rate = 'mean'
).round(3)

print('отток по оценке дохода: ')
print(churn_by_salary)

нет четкой зависимости между доходом и оттоком.

In [ ]:
corr = df['EstimatedSalary'].corr(df['Balance'])
corr

связи EstimatedSalary и Balance нет

In [ ]:
df.groupby('Salary_group', observed=False)['CreditScore'].mean().round(0)

Рейтинг практически совпадает у всех - значит, EstimatedSalary не отражает реальный статус.

In [ ]:
df.groupby('Salary_group', observed=False)['NumOfProducts'].mean().round(2)

Количество продуктов одинаковое

Анализ EstimatedSalary показал, что данный признак абсолютно неинформативен, его лучше не включать в модель.

# Биннинг и WOE

In [ ]:
#подготовка 

df_clean = df.copy()

In [ ]:
def calc_woe_iv(df, col, target = 'Exited'):

    #группируем по категории/бину
    woe_df = df.groupby(col)[target].agg(['count', 'sum'])
    woe_df.columns = ['total', 'churned']
    woe_df['non_churned'] = woe_df['total'] - woe_df['churned']

    #доли в группах
    total_non = woe_df['non_churned'].sum() #все оставшиеся
    total_churn = woe_df['churned'].sum() #все ушедшие

    #избегаем деления на 0
    woe_df['dist_non'] = woe_df['non_churned'] / (total_non + 10**(-6))
    woe_df['dist_churn'] = woe_df['churned'] / (total_churn + 10**(-6))

    #WOE
    woe_df['woe'] = np.log(woe_df['dist_non'] / woe_df['dist_churn'])

    #WOE
   # woe_df['woe'] = np.log((non_churned / total_non) / (churned / total_churn))
    
    #заменим бесконечности на 0 (если нет хороших/плохих)
    woe_df['woe'] = woe_df['woe'].replace([np.inf, -np.inf], 0)

    #IV
    woe_df['iv'] = (woe_df['dist_non'] - woe_df['dist_churn']) * woe_df['woe']
    iv = woe_df['iv'].sum()

    return woe_df[['woe']], iv


In [ ]:
#разделим данные на train/test

df_train, df_test = train_test_split(df_clean, test_size = 0.25, random_state = 42, stratify = df_clean['Exited'])

Биннинг числовых признаков

Проверим как разделились положительные Exited по выборкам

In [ ]:
print('Train Exited = 1: ', df_train['Exited'].mean())
print('Test Exited = 1:', df_test['Exited'].mean())

Разделение прошло идеально, пропорции практически идентичны.

### биннинг возраста

In [ ]:
plt.figure(figsize = (10,6))
sns.histplot(df_clean['Age'], bins = 30, kde = True, color = 'pink', edgecolor = 'purple')
plt.title('Распределение возраста клинтов')
plt.xlabel('Возраст')
plt.ylabel('Число клиентов')
plt.axvline(x = 25, color = 'red', linestyle = '--', alpha = 0.7)
plt.axvline(x = 35, color = 'red', linestyle = '--', alpha = 0.7)
plt.axvline(x = 45, color = 'red', linestyle = '--', alpha = 0.7)
plt.axvline(x = 56, color = 'red', linestyle = '--', alpha = 0.7)
plt.show()

In [ ]:
#посмотрим размеры выбранных групп:
bins = [18, 25, 35, 45, 56, 100]
labels = ['18-25', '26-35', '36-45', '46-56', '57+']
df['Age_bin_custom'] = pd.cut(df['Age'], bins = bins, labels = labels)
df['Age_bin_custom'].value_counts().sort_index()

Размер групп нормальный, во всех группах не менне 5% значений от всего датасета

In [ ]:
#Посмотрим отток по этим группам
churn_by_age = df.groupby('Age_bin_custom', observed=False)['Exited'].agg(
    count = 'size',
    churn_rate = 'mean'
).round(3)

churn_by_age

Отток растет с возрастом, особенно после 46 лет, группе 57+ отток немного снижается, но все равно высокий. Выберем такие границы для биннинга признака Age

In [ ]:
#биннинг для возраста
bins_age = [18, 25, 35, 45, 56, 100]
labels_age = ['18-30', '31-40', '41-50', '51-60', '60+']
df_train['Age_bin_custom'] = pd.cut(df_train['Age'], bins = bins_age, labels = labels_age, include_lowest = True)
df_test['Age_bin_custom'] = pd.cut(df_test['Age'], bins = bins_age, labels = labels_age, include_lowest = True)

#### Проверка биннинга признака Age с помощью библиотеки optbinning

In [ ]:
#посмотрим event rate для ручного биннинга
manual_stats = df_train.groupby('Age_bin_custom', observed=False)['Exited'].agg(
    count = 'size',
    event_rate = 'mean'
).reset_index()

manual_stats

In [ ]:
#создадим таблицу событий/не событий
manual_table = df_train.groupby('Age_bin_custom', observed=False)['Exited'].agg(
    non_event = lambda x: (x == 0).sum(),
    event = lambda x: (x == 1).sum()
).reset_index()

#добавляем доли
manual_table['total'] = manual_table['non_event'] + manual_table['event']
manual_table['pct_non_event'] = manual_table['non_event'] / manual_table['non_event'].sum()
manual_table['pct_event'] = manual_table['event'] / manual_table['event'].sum()

#вычисляем woe/iv
manual_table['woe'] = np.log(manual_table['pct_event'] / manual_table['pct_non_event'])
manual_table['iv'] = (manual_table['pct_event'] - manual_table['pct_non_event']) * manual_table['woe']

total_iv_manual = manual_table['iv'].sum()
total_iv_manual

In [ ]:
#проведем автоматический биннинг
optb = OptimalBinning(
    name = 'Age',
    dtype = 'numerical', #тип признака - числовой
    solver = 'cp', 
    min_bin_size = 0.1, #минимум 5% наблюдений в бине
    max_n_bins = 5, #максимум 10 бинов
    monotonic_trend = 'auto'
)

optb.fit(df_train['Age'], df_train['Exited'])

binning_table = optb.binning_table.build()
print('Таблица оптимального биннинга для Age')
print(binning_table)

Автоматический биннинг чуть лучше по информативности(IV = 0.4), чем ручной(IV = 0.42), что незначительно. Разница в IV небольшая(~0.02), поэтому лучше выбрать ручной биннинг, он интерпретируем, не вызовет вопросов у заказчика и качество модели почти не пострадает.

### биннинг по уровню баланса

In [ ]:
plt.figure(figsize = (10,6))
sns.histplot(df_clean['Balance'], bins = 30, kde = True, color = 'pink', edgecolor = 'purple')
plt.title('Распределение баланса клинтов')
plt.xlabel('Balance')
plt.ylabel('Число клиентов')
plt.axvline(x = 0, color = 'red', linestyle = '--', alpha = 0.7)
plt.axvline(x = 60000, color = 'red', linestyle = '--', alpha = 0.7)
plt.axvline(x = 110000, color = 'red', linestyle = '--', alpha = 0.7)
plt.axvline(x = 150000, color = 'red', linestyle = '--', alpha = 0.7)
plt.axvline(x = 200000, color = 'red', linestyle = '--', alpha = 0.7)
plt.axvline(x = 250000, color = 'red', linestyle = '--', alpha = 0.7)
plt.show()

In [ ]:
#посмотрим размеры выбранных групп:
bins = [0, 60000, 110000, 150000, 270000]
labels = ['0-60k', '60k-110k', '110k-150k', '150k-270k']
df_clean['Balance_bin_custom'] = pd.cut(df_clean['Balance'], bins = bins, labels = labels)
df_clean['Balance_bin_custom'].value_counts().sort_index()

Количество клиентов в бинах нормальное

In [ ]:
#Определим такие границы бинов:
bins_balance = [-1, 60000, 110000, 150000, 270001]
labels_balance = ['0-60k', '60k-110k', '110k-150k', '150k-270k']
df_train['Balance_bin_custom'] = pd.cut(df_train['Balance'], bins = bins_balance, labels = labels_balance, include_lowest = True, right = True)
df_test['Balance_bin_custom'] = pd.cut(df_test['Balance'], bins = bins_balance, labels = labels_balance, include_lowest = True, right = True)

#### Проверка биннинга признака Balance с помощью библиотеки optbinning

In [ ]:
#посмотрим event rate для ручного биннинга
manual_stats = df_train.groupby('Balance_bin_custom', observed=False)['Exited'].agg(
    count = 'size',
    event_rate = 'mean'
).reset_index()

manual_stats

In [ ]:
#создадим таблицу событий/не событий
manual_table = df_train.groupby('Balance_bin_custom', observed=False)['Exited'].agg(
    non_event = lambda x: (x == 0).sum(),
    event = lambda x: (x == 1).sum()
).reset_index()

#добавляем доли
manual_table['total'] = manual_table['non_event'] + manual_table['event']
manual_table['pct_non_event'] = manual_table['non_event'] / manual_table['non_event'].sum()
manual_table['pct_event'] = manual_table['event'] / manual_table['event'].sum()

#вычисляем woe/iv
manual_table['woe'] = np.log(manual_table['pct_event'] / manual_table['pct_non_event'])
manual_table['iv'] = (manual_table['pct_event'] - manual_table['pct_non_event']) * manual_table['woe']

total_iv_manual = manual_table['iv'].sum()
total_iv_manual

In [ ]:
#проведем автоматический биннинг
optb = OptimalBinning(
    name = 'Balance',
    dtype = 'numerical', #тип признака - числовой
    solver = 'cp', 
    min_bin_size = 0.1, #минимум 5% наблюдений в бине
    max_n_bins = 5, #максимум 10 бинов
    monotonic_trend = 'auto'
)

optb.fit(df_train['Balance'], df_train['Exited'])

binning_table = optb.binning_table.build()
print('Таблица оптимального биннинга для Balance')
print(binning_table)

Значение IV для автоматического биннинга выше, чем значение ручного, в дальнейшем будем использовать автоматический.

Вычислим woe для признака balance с помощью библиотеки optbinning

In [ ]:
df_train['Balance_woe_optb'] = optb.transform(df_train['Balance'], metric = 'woe')
df_test['Balance_woe_optb'] = optb.transform(df_test['Balance'], metric = 'woe')

### биннинг по уровню рейтинга

In [ ]:
plt.figure(figsize = (10,6))
sns.histplot(df_clean['CreditScore'], bins = 30, kde = True, color = 'pink', edgecolor = 'purple')
plt.title('Распределение CreditScore')
plt.xlabel('CreditScore')
plt.ylabel('Число клиентов')
plt.axvline(x = 300, color = 'red', linestyle = '--', alpha = 0.7)
plt.axvline(x = 580, color = 'red', linestyle = '--', alpha = 0.7)
plt.axvline(x = 670, color = 'red', linestyle = '--', alpha = 0.7)
plt.axvline(x = 740, color = 'red', linestyle = '--', alpha = 0.7)
plt.axvline(x = 850, color = 'red', linestyle = '--', alpha = 0.7)
plt.show()

In [ ]:
bins_credit = [300, 580, 670, 740, 850]
labels_credit = ['Low', 'Fair', 'Good','Excellent']
df_train['CreditScore_bin'] = pd.cut(df_train['CreditScore'], bins = bins_credit, labels = labels_credit)
df_test['CreditScore_bin'] = pd.cut(df_test['CreditScore'], bins = bins_credit, labels = labels_credit)

In [ ]:
#посмотрим размеры выбранных групп:
bins = [300, 580, 670, 740, 850]
labels = ['Low', 'Fair', 'Good','Excellent']
df_clean['CreditScore_bin'] = pd.cut(df_clean['CreditScore'], bins = bins, labels = labels)
df_train['CreditScore_bin'].value_counts().sort_index()

В группе Fair почти половина, остальные группы достаточно большие

## Рассчет WOE/IV

In [ ]:
features_final = ['Age_bin_custom', 'NumOfProducts', 'Geography', 'IsActiveMember', 'Complain']

In [ ]:
woe_dict = {}
iv_dict = {}

features_for_woe = [
    'Age_bin_custom', 'CreditScore_bin', 'NumOfProducts', 'Geography', 'IsActiveMember', 'Complain'
]
for feature in features_for_woe:
    woe, iv = calc_woe_iv(df_train, feature, 'Exited')
    woe_dict[feature] = woe['woe'].to_dict()
    iv_dict[feature] = iv
    print(f'{feature}: IV = {iv:.3f}')

Age_bin_custom: IV = 0.402 ---> ✔сильный - очень информативен

Balance_bin_custom: IV = 0.063 ---> ⚪очень слабый - почти неинформативен

CreditScore_bin: IV = 0.004 ---> почти бесполезный - можно исключить

NumOfProducts: IV = 0.388 ---> ✔хороший - важный признак

Geography: IV = 0.107 ---> ⚪ может быть информативным

IsActiveMember: IV = 0.082 ---> ⚪слабый, но может быть полезным

Complain: IV = 3.168 ---> ❓аномально высокий, требует внимания

Высокое значение IV = 3.168 обусловлено экстремально сильной связью:

- после жалобы клиент уходит в 71% случаев

- без жалобы только в 7%
это не ошибка, а сигнал: обработка жалоб - ключевая проблема оттока. Признак оставлен без изменений, он отражает реальную бизнес-рискованность



In [ ]:
df_train_model = df_train.copy()
df_test_model = df_test.copy()

for feature in woe_dict:
    new_col = f'{feature}_woe'

    #приводим к строке, чтобы избежать проблем с category
    df_train_model[feature] = df_train_model[feature].astype(str)
    df_test_model[feature] = df_test_model[feature].astype(str)
    #заполняем NaN нулем (нейтральное состояние woe)
    df_train_model[new_col] = df_train_model[feature].map(woe_dict[feature]).fillna(0)
    df_test_model[new_col] = df_test_model[feature].map(woe_dict[feature]).fillna(0)

In [ ]:
#переcчитаем woe вручную для Сomplain_Yes
group = df_train.groupby('Complain')['Exited'].agg(['count', 'sum'])
group.columns = ['total', 'churned']
group['non_churned'] = group['total'] - group['churned']

total_non = group['non_churned'].sum()
total_churn = group['churned'].sum()

group['dist_non'] = group['non_churned'] / total_non
group['dist_churn'] = group['churned'] / total_churn

group['woe'] = np.log(group['dist_non'] / group['dist_churn'])
group['iv'] = (group['dist_non'] - group['dist_churn']) * group['woe']
iv = group['iv'].sum()

print(f'реальный {iv:.3f}')

Рассчитанный вручную woe равен рассчитанному woe по формуле

# Подготовка и обучение модели логистической регрессии

In [ ]:
selected_cols = [f'{feat}_woe' for feat in features_final]
selected_cols.append('Balance_woe_optb')

In [ ]:
# применим woe к train и  test
df_train_model = df_train.copy()
df_test_model = df_test.copy()

for feature in woe_dict:
    new_col = f'{feature}_woe'

    #df_train_model[feature] = df_train_model[feature].astype(str)
    #df_test_model[feature] = df_test_model[feature].astype(str)
    
    df_train_model[new_col] = df_train_model[feature].map(woe_dict[feature])
    df_test_model[new_col] = df_test_model[feature].map(woe_dict[feature])

In [ ]:
#подгогтовим Х и У
selected_cils = [f'{feat}_woe' for feat in features_final]

X_train = df_train_model[selected_cols]
X_test = df_test_model[selected_cols]
y_train = df_train_model['Exited']
y_test = df_test_model['Exited']

In [ ]:
X_train.isnull().sum()

In [ ]:
#обучим логистическую регрессию

model = LogisticRegression(class_weight = 'balanced', random_state = 42)
model.fit(X_train, y_train)

#прогноз вероятности
y_pred_proba = model.predict_proba(X_test)[:,1]

#AUC-ROC
auc = roc_auc_score(y_test, y_pred_proba)
print(f'AUC-ROC: {auc:.3f}')

Получилась очень высокая способность модели

In [ ]:
coefficients = pd.DataFrame({
    'Feature': selected_cols,
    'Coeffitient': model.coef_[0]
}).round(3)

print('Коэффициенты модели:')
print(coefficients)

Отрицательный woe - высокий риск ухода, модель "говорит": чем ниже это число (woe), тем выше шанс ухода и ставит отрицательный коэффициент

Например,

- Клиент 1, жаловался,

Complain_Yes_woe = -2.29

-0.93 * -2.29 = + 2.13 ---> увеличивает вероятность ухода 


- Клиент 2, не жаловался

Complain_Yes_woe = 1.22

-0.93 * 1.22 = -1.14 ---> уменьшает вероятность ухода 
Отрицательный коэффициент + отрицательный woe = + = риск ухода растет

In [ ]:
#простая кросс-валидация

cv_auc = cross_val_score(model, X_train, y_train, cv = 5, scoring = 'roc_auc')
print(f'CV AUC : {cv_auc.mean():.3f} +- {cv_auc.std():.3f}')

Средний AUC по 5 фолдам 0.901, почти совпадает с AUC test(0.895), значит модель не переобучилась,

+-0.004 - очень маленькое стандартное отклонение ---> модель стабильна на разных частях данных, результат не случаен, она действительно понимает закономерности

In [ ]:
y_pred_proba = model.predict_proba(X_test)[:, 1]

#бинарные предсказания: если вероятностью > 0.5 --> уйдет
y_pred = (y_pred_proba > 0.5).astype(int)

In [ ]:
#Матрица ошибок

#матрица
cm = confusion_matrix(y_test, y_pred)

#визуализация
plt.figure(figsize = (6,4))
sns.heatmap(cm, annot = True, fmt = 'd', cmap = 'Blues', 
           xticklabels = ['Остались', 'Ушли'],
           yticklabels = ['Остались', 'Ушли'])
plt.title('Матрица ошибок')
plt.xlabel('Предсказано')
plt.ylabel('Реальность')
plt.show()

In [ ]:
print(classification_report(y_test, y_pred, target_names = ['Остался', 'Ушел']))

Найдем гиперпараметры

In [ ]:
%%time
param_grid = {
    'C': [0.01, 0.1, 1, 10],
    'penalty': ['l1', 'l2'], 
    'solver': ['liblinear'],
    'max_iter': [1000],
    'class_weight': ['balanced']
}

log_reg = LogisticRegression(random_state = 42)

grid_search_lr = GridSearchCV(
    estimator = log_reg,
    param_grid = param_grid,
    scoring = 'roc_auc',
    cv = 3,
    verbose = 1,
    n_jobs = -1
)

grid_search_lr.fit(X_train, y_train)

print('лучшие параметры: ', grid_search_lr.best_params_)
print('лучший auc: ', grid_search_lr.best_score_.round(3))

In [ ]:
#создаем модель с лучшими параметрами
final_lr_model = LogisticRegression(
    C = 1,
    class_weight = 'balanced',
    max_iter = 1000,
    penalty = 'l2',
    solver = 'liblinear',
    random_state = 42
)

#обучаем на поллных train
final_lr_model.fit(X_train, y_train)

#предсказываем на test
y_pred_proba_lr_final = final_lr_model.predict_proba(X_test)[:,1]
y_pred_lr_final = (y_pred_proba_lr_final > 0.5).astype(int)

#оценка

print('final LR AUC:', roc_auc_score(y_test, y_pred_proba_lr_final))
print('\nClassification Report:')
print(classification_report(y_test, y_pred_lr_final))

In [ ]:
#простая кросс-валидация

cv_auc = cross_val_score(final_lr_model, X_train, y_train, cv = 5, scoring = 'roc_auc')
print(f'CV AUC : {cv_auc.mean():.3f} +- {cv_auc.std():.3f}')

# Подготовка и обучение модели случайный лес

In [ ]:
features_final_rf = ['Age', 'Balance', 'NumOfProducts', 'Complain', 'CreditScore']

In [ ]:
X_train_rf = df_train[features_final_rf].copy()
X_test_rf = df_test[features_final_rf].copy()
y_train_rf = df_train['Exited'].copy()
y_test_rf = df_test['Exited'].copy()

Случайный лес чувствителен к лишним признакам, посмотрим какие признаки важны для модели и включим их в выборку

In [ ]:
model = RandomForestClassifier(class_weight = 'balanced', random_state = 42)
model.fit(X_train_rf, y_train_rf)

importances = model.feature_importances_
feature_names = X_train_rf.columns

feat_importances = pd.Series(importances, index = feature_names).sort_values(ascending = False)
print(feat_importances)

In [ ]:
#проверим на пропуски
if X_train_rf.isnull().sum().sum() > 0:
    print('есть NaN в X_train_rf - заполняем')
    X_train_rf = X_train_rf.fillna(-1)
if X_test_rf.isnull().sum().sum() > 0:
    print('есть NaN в X_test_rf - заполняем')
    X_test_rf = X_test_rf.fillna(-1)

In [ ]:
#обучим случайный лес

#модель
rf_model = RandomForestClassifier(
    n_estimators = 100,
    max_depth = 8,
    min_samples_split = 10,
    min_samples_leaf = 5,
    random_state = 42,
    n_jobs = -1
)

#обучение
rf_model.fit(X_train_rf, y_train)

#прогноз вероятности
y_pred_proba_rf = rf_model.predict_proba(X_test_rf)[:, 1]

#AUC
auc_rf = roc_auc_score(y_test, y_pred_proba_rf)
print(f'AUC-ROC (Random Forest): {auc_rf:.3f}')

AUC-ROC получился достаточно большим

In [ ]:
feat_importance = pd.DataFrame({
    'Feature': X_train_rf.columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending = False)

print('\nВажность признаков:')
print(feat_importance)

#график
plt.figure(figsize = (8, 6))
top10 = feat_importance.head(10)
plt.barh(top10['Feature'], top10['Importance'])
plt.title('важность признаков в Random Forest')
plt.xlabel('важность')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

признак Complain оказывает очень большое влияние на модель

In [ ]:
#бинарный прогноз 
y_pred_rf = (y_pred_proba_rf > 0.2).astype(int)

print('\n Classification Report(RF)')
print(classification_report(y_test_rf, y_pred_rf))

In [ ]:
#Матрица ошибок

#матрица
cm = confusion_matrix(y_test_rf, y_pred_rf)

#визуализация
plt.figure(figsize = (6,4))
sns.heatmap(cm, annot = True, fmt = 'd', cmap = 'Blues', 
           xticklabels = ['Остались', 'Ушли'],
           yticklabels = ['Остались', 'Ушли'])
plt.title('Матрица ошибок(RF)')
plt.xlabel('Предсказано')
plt.ylabel('Реальность')
plt.show()

Найдем гиперпараметры

In [ ]:
%%time
param_grid = {
    'n_estimators': [100,200],
    'max_depth': [8, 10, 12], 
    'min_samples_split': [10,20],
    'min_samples_leaf': [5,10],
    'class_weight': ['balanced']
}

rf = RandomForestClassifier(random_state = 42, n_jobs = -1)

grid_search = GridSearchCV(
    estimator = rf,
    param_grid = param_grid,
    scoring = 'roc_auc',
    cv = 3,
    verbose = 1,
    n_jobs = -1
)

grid_search.fit(X_train_rf, y_train_rf)

print('лучшие параметры: ', grid_search.best_params_)
print('лучший auc: ', grid_search.best_score_.round(3))

In [ ]:
y_pred_tuned = (y_pred_proba_rf > 0.2).astype(int)

print(classification_report(y_test_rf, y_pred_tuned))

Нашли гиперпараметры, теперь обучим финальную модель с этими параметрами на полных данных train

In [ ]:
%%time
#создаем модель с лучшими параметрами
final_rf_model = RandomForestClassifier(
    class_weight = 'balanced',
    max_depth = 8,
    min_samples_leaf = 5,
    min_samples_split = 20, 
    n_estimators = 200,
    random_state = 42,
    n_jobs = -1
)

#обучаем на поллных train
final_rf_model.fit(X_train_rf, y_train_rf)

#предсказываем на test
y_pred_proba_final = final_rf_model.predict_proba(X_test_rf)[:,1]
y_pred_final = (y_pred_proba_final > 0.5).astype(int)

#оценка

print('final RF AUC:', roc_auc_score(y_test_rf, y_pred_proba_final))
print('\nClassification Report:')
print(classification_report(y_test_rf, y_pred_final))

In [ ]:
%%time
#простая кросс-валидация

cv_auc = cross_val_score(final_rf_model, X_train, y_train, cv = 5, scoring = 'roc_auc')
print(f'CV AUC : {cv_auc.mean():.3f} +- {cv_auc.std():.3f}')

# Калибровка RF и логистической регрессии

In [ ]:
#ДЛЯ ЛОГ РЕГ
prob_true_lr, prob_pred_lr = calibration_curve(
    y_test, y_pred_proba, n_bins = 10
)

#для RF
prob_true_rf, prob_pred_rf = calibration_curve(
    y_test, y_pred_proba_rf, n_bins = 10
)

plt.figure(figsize = (8,6))
plt.plot(prob_pred_lr, prob_true_lr, marker = 'o', label = 'логистическая регрессия')
plt.plot(prob_pred_rf, prob_true_rf, marker = 's', label = 'RF')
plt.plot([0,1], [0,1], linestyle = '--', color = 'gray', label = 'идеальная калибровка')
plt.xlabel('средняя прогнозируемая вер-ть')
plt.ylabel('доля истинных уходов')
plt.title('вероятностная кривая')
plt.legend()
plt.grid(True, alpha = 0.3)
plt.show()

Кривые моделей отклоняются от идеальной калиброки, попробуем откалибровать их.

In [ ]:
#калибруем логистическую регрессию
calibrated_lr = CalibratedClassifierCV(final_lr_model, method = 'isotonic', cv = 3)
calibrated_lr.fit(X_train, y_train)

#калибруем RF
calibrated_rf = CalibratedClassifierCV(final_rf_model, method = 'isotonic', cv = 3)
calibrated_rf.fit(X_train_rf, y_train_rf)

Построим новые вероятностные кривые

In [ ]:
#прогнозы после калибровки
y_proba_cal_lr = calibrated_lr.predict_proba(X_test)[:, 1]
y_proba_cal_rf = calibrated_rf.predict_proba(X_test_rf)[:, 1]

#ДЛЯ ЛОГ РЕГ
prob_true_lr, prob_pred_lr = calibration_curve(
    y_test, y_proba_cal_lr, n_bins = 10
)

#для RF
prob_true_rf, prob_pred_rf = calibration_curve(
    y_test, y_proba_cal_rf, n_bins = 10
)

plt.figure(figsize = (8,6))
plt.plot(prob_pred_lr, prob_true_lr, marker = 'o', label = 'логистическая регрессия(откалиброванная)')
plt.plot(prob_pred_rf, prob_true_rf, marker = 's', label = 'RF(откалиброванная )')
plt.plot([0,1], [0,1], linestyle = '--', color = 'gray', label = 'идеальная калибровка')
plt.xlabel('средняя прогнозируемая вер-ть')
plt.ylabel('доля истинных уходов')
plt.title('вероятностная кривая')
plt.legend()
plt.grid(True, alpha = 0.3)
plt.show()

Изменений в лучшую сторону не видно, попробуем калибровку сигмоидой

In [ ]:
#калибруем через Platt Scaling

#для RF
calibrated_rf_platt = CalibratedClassifierCV(final_rf_model, method = 'sigmoid', cv = 3)
calibrated_rf_platt.fit(X_train_rf, y_train_rf)

#для LR
calibrated_lr_platt = CalibratedClassifierCV(final_lr_model, method = 'sigmoid', cv = 3)
calibrated_lr_platt.fit(X_train, y_train)

#прогноз RF
y_proba_rf_platt = calibrated_rf_platt.predict_proba(X_test_rf)[:, 1]

#прогноз LR
y_proba_lr_platt = calibrated_lr_platt.predict_proba(X_test)[:, 1]

#калибровочная кривая RF
prob_true_platt_rf, prob_pred_platt_rf = calibration_curve(y_test_rf, y_proba_rf_platt, n_bins = 10)

#калибровочная кривая LR
prob_true_platt_lr, prob_pred_platt_lr = calibration_curve(y_test, y_proba_lr_platt, n_bins = 10)

#график
plt.figure(figsize = (8,6))
plt.plot(prob_pred_platt_lr, prob_true_platt_lr, marker = 'o', label = 'логистическая регрессия(обкалиброванная sigmoid)')
plt.plot(prob_pred_platt_rf, prob_true_platt_rf, marker = 's', label = 'RF(откалиброванная sigmoid)')
plt.plot([0,1], [0,1], linestyle = '--', color = 'gray', label = 'идеальная калибровка')
plt.xlabel('средняя прогнозируемая вер-ть')
plt.ylabel('доля истинных уходов')
plt.title('вероятностная кривая')
plt.legend()
plt.grid(True, alpha = 0.3)
plt.show()

Посмотрим значения Brier Score

In [ ]:
#получаем вероятности до и после калибровки
probs_uncalibrated = final_rf_model.predict_proba(X_test_rf)[:, 1]
probs_sigmoid = calibrated_rf_platt.predict_proba(X_test_rf)[:, 1]
probs_isotonic = calibrated_rf.predict_proba(X_test_rf)[:, 1]
#считаем Brier Score
brier_uncal = brier_score_loss(y_test_rf, probs_uncalibrated)
brier_sig = brier_score_loss(y_test_rf, probs_sigmoid)
brier_iso = brier_score_loss(y_test_rf, probs_isotonic)

print(f'Uncalibrated: {brier_uncal:.4f}')
print(f'Sigmoid: {brier_sig:.4f}')
print(f'Isotonic: {brier_iso:.4f}')

In [ ]:
#получаем вероятности до и после калибровки
probs_uncalibrated = final_lr_model.predict_proba(X_test)[:, 1]
probs_sigmoid = calibrated_lr_platt.predict_proba(X_test)[:, 1]
probs_isotonic = calibrated_lr.predict_proba(X_test)[:, 1]
#считаем Brier Score
brier_uncal = brier_score_loss(y_test, probs_uncalibrated)
brier_sig = brier_score_loss(y_test, probs_sigmoid)
brier_iso = brier_score_loss(y_test, probs_isotonic)

print(f'Uncalibrated: {brier_uncal:.4f}')
print(f'Sigmoid: {brier_sig:.4f}')
print(f'Isotonic: {brier_iso:.4f}')

Platt Scaling хуже, чем isotonic

Для обеих моделей выбираем калибровку Isotonic. Сохраним их

# Сохранение

In [ ]:
#для RF

#выберем лучшую калибровку
best_calibrated_rf = calibrated_rf

#сохраняем
import joblib
joblib.dump(best_calibrated_rf, 'best_calibrated_rf_not_anomalous.pkl')
#используем
best_calibrated_rf = joblib.load('best_calibrated_rf_not_anomalous.pkl')

#прогноз
y_pred_proba = best_calibrated_rf.predict_proba(X_test_rf)[:, 1]

######


In [ ]:
#для LR
#выберем лучшую калибровку
best_calibrated_lr = calibrated_lr

#сохраняем
import joblib
joblib.dump(best_calibrated_lr, 'best_calibrated_lr_not_anomalous.pkl')
#используем
best_calibrated_lr = joblib.load('best_calibrated_lr_not_anomalous.pkl')

#прогноз
y_pred_proba = best_calibrated_lr.predict_proba(X_test)[:, 1]

# Вывод

Построение моделей на полностью очищенном от аномалий и пропусков датасете показало лучшие результаты AUC-ROC и кросс-валидации, чем на моделях без удаления данных,  модели получились стабильными, но калибровка данных моделей хуже. 

Для модели логистическая регрессия, AUC-ROC = 0.895(CV AUC : 0.902 +- 0.005) , для модели случайный лес, AUC-ROC = 0.93(CV AUC : 0.923 +- 0.002).

Метрики:

    Для логистической регрессии:
        precision    recall
    0     0.95        0.95
    1     0.79        0.79


    Для случайного леса:
        precision    recall
    0     0.95        0.95
    1     0.79        0.80

Значимых различий нет.



Пусть первый способ это построение моделей с созданием флагов и обработкой пропусков на этапе предобработки. Второй случай это построение моделей с предварительным удалением наблюдений с аномалиями и пропущенными значениями.

Первый способ: лучше если важны интерпретируемые вероятности(для принятия решений, порогов, рисков).

Второй способ: лучше с точки зрения ранжирования, но хуже для интерпретации вероятностей.

Задача, поставленная в тз - прогнозировать вероятность ухода клиента, значит важны интерпретируемые вероятности, а не только ранжирование.

Исходя из задачи, первый способ лучше, у него хорошая калибровка, дает достоверные вероятности и подходит под постановку вопроса "какова вероятность ухода?".